<style>
.mermaid {
  width: 100%;
  overflow-x: auto;
  padding: 1.25rem 0 1.75rem;
}
.mermaid svg {
  width: 100% !important;
  max-width: 1120px !important;
  min-width: 0 !important;
  height: auto !important;
  display: block;
  margin: 0 auto;
}
</style>

# Architecture Diagram

## Mục tiêu

Phần sơ đồ kiến trúc tổng hợp thiết kế hoàn chỉnh của Lab 04 dựa trên đề bài `[BigData] Lab04 - StreamingV0.pdf` và hiện trạng mã nguồn trong repository. Trọng tâm là pipeline streaming tăng dần: Parser Service publish event qua Kafka, Neo4j Kafka Sink ghi graph topology trực tiếp bằng Cypher `MERGE`, và Spark Structured Streaming ghi metadata sang MongoDB bằng upsert có checkpoint.

## Yêu cầu từ đề bài

| Nhóm yêu cầu | Cách hệ thống đáp ứng |
|---|---|
| Repository cloning và file discovery | Shallow clone repository `huggingface/transformers-pr-agent`, sau đó lọc các file Python hợp lệ. |
| Incremental CPG Parser Service | Parser xử lý từng file độc lập, dùng `ast` chuẩn của Python để sinh AST, CFG, DFG và Call graph. |
| Kafka topic layout | Tách riêng `cpg.nodes`, `cpg.edges`, `source.metadata`, `parser.errors` và `connector.errors`. |
| Neo4j graph ingestion | Kafka Connect Sink đọc node/edge events và ghi graph trực tiếp vào Neo4j bằng Cypher `MERGE`, kèm uniqueness constraints theo stable ID. |
| MongoDB metadata ingestion | Spark Structured Streaming consume `source.metadata`, dùng checkpoint và upsert document theo `file_id`. |
| Idempotent replay | Stable deterministic IDs, SQLite state store, graph diff, Neo4j `MERGE`/delete idempotent và MongoDB upsert tạo replay không trùng lặp. |

## Sơ đồ kiến trúc tổng thể

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontFamily": "Inter, Arial, sans-serif",
    "fontSize": "22px",
    "primaryColor": "#eef2ff",
    "primaryBorderColor": "#4f46e5",
    "primaryTextColor": "#111827",
    "lineColor": "#334155",
    "clusterBkg": "#f8fafc",
    "clusterBorder": "#cbd5e1"
  },
  "flowchart": {
    "htmlLabels": true,
    "nodeSpacing": 58,
    "rankSpacing": 76,
    "curve": "basis"
  }
}}%%
flowchart TB
    Source["Source repo<br/><b>transformers-pr-agent</b>"]
    Discovery["Discovery CLI<br/>lọc file .py"]
    Parser["Parser Service<br/>xử lý từng file"]

    Source --> Discovery --> Parser

    subgraph Core["Parser core"]
        Ast["AST"]
        Cfg["CFG"]
        Dfg["DFG"]
        Calls["Call graph"]
        Meta["Metadata"]
        Stable["Stable IDs"]
        State[("SQLite state")]
    end

    Parser --> Core
    Core --> Events["Event envelope<br/>schema_version + event_time"]

    subgraph Kafka["Kafka topics"]
        Nodes["cpg.nodes"]
        Edges["cpg.edges"]
        Metadata["source.metadata"]
        ParserErrors["parser.errors"]
        ConnectorErrors["connector.errors"]
    end

    Events --> Nodes
    Events --> Edges
    Events --> Metadata
    Events --> ParserErrors

    subgraph GraphPath["Graph ingestion"]
        Neo4jSink["Neo4j Kafka Sink"]
        Neo4j[("Neo4j graph")]
    end

    Nodes --> Neo4jSink
    Edges --> Neo4jSink
    Neo4jSink -->|"MERGE / idempotent delete"| Neo4j
    Neo4jSink -.-> ConnectorErrors

    subgraph MetadataPath["Metadata ingestion"]
        Spark["Spark Streaming"]
        Mongo[("MongoDB metadata")]
    end

    Metadata --> Spark
    Spark -->|"checkpoint + upsert"| Mongo
```

## Luồng replay tăng dần

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontFamily": "Inter, Arial, sans-serif",
    "fontSize": "22px",
    "primaryColor": "#eef2ff",
    "primaryBorderColor": "#4f46e5",
    "primaryTextColor": "#111827",
    "lineColor": "#334155",
    "clusterBkg": "#f8fafc",
    "clusterBorder": "#cbd5e1"
  },
  "flowchart": {
    "htmlLabels": true,
    "nodeSpacing": 58,
    "rankSpacing": 76,
    "curve": "basis"
  }
}}%%
flowchart TB
    Change["1. Sửa một file Python"]
    Replay["2. Chạy replay-file"]
    LoadState["3. Đọc state cũ<br/>content_hash + graph IDs"]
    Parse["4. Parse nội dung mới"]
    Diff["5. So sánh graph cũ và mới"]

    Change --> Replay --> LoadState --> Parse --> Diff

    Diff --> Delete["6a. Sinh DELETE events<br/>cho phần tử cũ"]
    Diff --> Upsert["6b. Sinh UPSERT events<br/>cho graph hiện tại"]
    Diff --> FileMeta["6c. Sinh metadata mới"]

    Delete --> Kafka["7. Publish vào Kafka"]
    Upsert --> Kafka
    FileMeta --> Kafka

    Kafka --> Ack["8. Kafka Acknowledgement"]
    Ack --> Commit["9. SQLite commit<br/>state mới (Không đợi downstream)"]

    Ack --> Downstream["10. Downstream (Bất đồng bộ)"]
    Downstream --> Neo4j["10a. Neo4j cập nhật graph<br/>không tạo duplicate"]
    Downstream --> Spark["10b. Spark đọc metadata<br/>theo checkpoint"]
    Spark --> Mongo["11. MongoDB upsert<br/>theo file_id"]
```

## Ranh giới kiến trúc

| Layer | Trách nhiệm | Quy tắc phụ thuộc |
|---|---|---|
| `domain/` | Model, enum, event contract và lỗi nghiệp vụ. | Không phụ thuộc layer khác. |
| `parsing/` | AST, CFG, DFG, Call graph, Stable ID và diff. | Chỉ phụ thuộc `domain`. |
| `application/` | Use case service và port interface. | Giao tiếp qua ports, không khởi tạo adapter cụ thể. |
| `infrastructure/` | Kafka, JSONL writer, SQLite, config và filesystem adapters. | Implement ports từ application. |
| `cli/` | Composition root, load config và inject adapters. | Được phép nối các layer khi chạy command. |
| `spark_jobs/` | Spark Structured Streaming job cho MongoDB. | Độc lập với parser core. |

## Ranh giới kiến trúc và Các hạn chế được chấp nhận (Architecture Boundaries & Accepted Limitations)

Hệ thống tuân thủ các ranh giới kiến trúc và chấp nhận các giới hạn kỹ thuật sau:
- **Không có distributed transaction giữa Kafka và SQLite**: Parser Service thực hiện commit SQLite sau khi nhận Ack từ Kafka broker mà không chờ các downstream databases hoàn thành.
- **Không bảo đảm thứ tự chéo topic**: Kafka chỉ đảm bảo thứ tự trong cùng partition của một topic. Sự xáo trộn giữa các topic được xử lý downstream (ví dụ: Neo4j tự tạo placeholder node khi edge đến trước).
- **Nhạy cảm thứ tự đến với cross-generation upserts**: Không có global monotonic generation sequence, do đó các thay đổi lịch sử chéo thế hệ phụ thuộc vào thứ tự sự kiện đến.
- **Tombstones kiểm soát thế hệ (generation-aware tombstones)**: Ngăn chặn stale resurrection trong Neo4j bằng generation identifier `file_id:content_hash:parser_version:schema_version`.
- **Kháng trùng lặp bằng upsert và checkpoint**: Spark Structured Streaming duy trì offset bằng checkpoint, và MongoDB thực hiện replace/upsert theo `file_id` để kháng trùng lặp.
- **Không đảm bảo exactly-once toàn bộ đầu cuối**: Hệ thống không cam kết exactly-once phân tán, mà hướng đến tính idempotent (kháng trùng lặp) đầu cuối.

## Reflection

Sơ đồ kiến trúc cho thấy Lab 04 không chỉ là một parser cục bộ mà là pipeline streaming nhiều hệ thống. Điểm quan trọng nhất là tách graph events và metadata events thành hai nhánh ingestion khác nhau: Neo4j nhận topology trực tiếp qua Kafka Connect, còn MongoDB nhận metadata qua Spark để tận dụng checkpoint. Khi Task 4 đã hoàn tất, nhánh Neo4j có thể được xem là consumer idempotent của graph events, phù hợp với yêu cầu replay không sinh duplicate.